# Background

Mental health issues have become increasingly prevalent, especially among students and young adults. Social media and online discussion platforms contain valuable textual information that can be analyzed to understand emotional conditions and stress levels.

This project develops a multitask Natural Language Processing (NLP) model capable of simultaneously classifying emotions and predicting stress levels from Indonesian text. To improve class balance and model generalization, synthetic data generated by Large Language Models (LLMs) is incorporated into the training process.

# Problem Statement

Many text datasets related to mental health suffer from class imbalance, where some emotional categories appear significantly more frequently than others. This imbalance can negatively impact model performance and bias predictions toward majority classes.

Additionally, identifying both emotional state and stress level typically requires separate models, increasing computational complexity and deployment costs.

# Objectives

The objectives of this project are:

1. Build a multitask NLP model for emotion classification and stress prediction.
2. Utilize IndoBERT to capture contextual understanding of Indonesian text.
3. Improve class distribution using synthetic data generated by LLMs.
4. Evaluate model performance using classification and regression metrics.

# Business Questions

1. What emotions are most frequently expressed in the dataset?
2. Can textual information be used to accurately predict stress levels?
3. Does synthetic data help reduce class imbalance?
4. How well can IndoBERT classify emotions from Indonesian text?
5. What relationship exists between emotional categories and stress levels?

#Install Library

In [4]:
# !pip install swifter

#Import Library

In [5]:
import pandas as pd
import numpy as np
import re
import string
import swifter

c:\Users\raiha\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#Load Dataset

In [6]:
df_epo = pd.read_csv('https://raw.githubusercontent.com/raihanmeintaro/Dataset/refs/heads/main/NLP/V2/df_epo_raw.csv')
df_llm = pd.read_csv('https://raw.githubusercontent.com/raihanmeintaro/Dataset/refs/heads/main/NLP/V2/df_llm_raw.csv')

##Synthetic Allocation

In [7]:
synthetic_count = int(len(df_epo) * 0.30)

print(f"Total synthetic used: {synthetic_count}")

Total synthetic used: 1896


In [8]:
df_combined = pd.concat([df_epo, df_llm.sample(n=synthetic_count, random_state=42)], ignore_index=True)

print(df_combined.shape)

(8216, 4)


In [9]:
max_count = df_combined['emotion_label'].value_counts().max()

balanced_data = []

for label in df_combined['emotion_label'].unique():
    subset = df_combined[df_combined['emotion_label'] == label]
    balanced_subset = subset.sample(n=max_count, replace=True, random_state=42)
    balanced_data.append(balanced_subset)

df_balanced = pd.concat(balanced_data,ignore_index=True)

df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced.shape)

print(df_balanced['emotion_label'].value_counts())

(11855, 4)
emotion_label
sad        2371
happy      2371
angry      2371
neutral    2371
anxious    2371
Name: count, dtype: int64


In [10]:
df = df_balanced.copy()

##Shuffling Dataset

In [11]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

##Missing Values Handling

In [12]:
df = df.dropna(subset=['text', 'emotion_label', 'stress_label'])
df = df.reset_index(drop=True)

print(df.isnull().sum())

Unnamed: 0       0
text             0
emotion_label    0
stress_label     0
dtype: int64


#Load Dictionary

##Load Kamus Alay from GitHub

In [13]:
alay_df = pd.read_csv('https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv')

# Buat Dictionary
alay_dict = dict(zip(alay_df['slang'], alay_df['formal']))

print("Kamus alay:", len(alay_dict))

Kamus alay: 4331


##Load Additional Kamus Alay

In [14]:
new_alay_df = pd.read_csv('https://raw.githubusercontent.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection/master/new_kamusalay.csv', encoding='latin-1', names=['slang', 'formal'])

# Buat Dictionary
new_alay_dict = dict(zip(new_alay_df['slang'], new_alay_df['formal']))

print("Typo dictionary:", len(new_alay_dict))

Typo dictionary: 15167


##Merge Kamus Alay Dictionary

In [15]:
combined_dict = {}

combined_dict.update(alay_dict)

combined_dict.update(new_alay_dict)

print("Total combined dictionary:", len(combined_dict))

Total combined dictionary: 15625


##Mapping Emoji Dictionary

In [16]:
emoji_dict = {

    "😭": " sedih ",
    "😢": " sedih ",
    "😔": " sedih ",

    "😡": " marah ",
    "😠": " marah ",

    "😊": " senang ",
    "😁": " senang ",
    "😄": " senang ",

    "😨": " takut ",
    "😰": " cemas ",
    "😥": " cemas "
}

#Pre-Processing

##Casefolding (lower text)

In [17]:
def case_folding(text):

    return text.lower()

##Cleaning Text

In [18]:
def clean_text(text):

    # remove placeholder
    text = re.sub(r'\[.*?\]', ' ', text)

    # remove url
    text = re.sub(r"http\S+", " ", text)

    text = re.sub(r"www\S+", " ", text)

    # remove html
    text = re.sub(r"<.*?>", " ", text)

    # remove mention
    text = re.sub(r"@\w+", " ", text)

    # remove hashtag
    text = re.sub(r"#\w+", " ", text)

    # remove number
    text = re.sub(r"\d+", " ", text)

    # normalize laugh
    text = re.sub(r'(wkwk+|haha+|hehe+)',' lucu ', text)

    # normalize repeated char
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # remove punctuation
    text = re.sub(
        r'[%s]' % re.escape(string.punctuation), ' ', text)

    # remove non alphabet
    text = re.sub(r'[^a-zA-Z\s]',' ', text)

    # remove extra whitespace
    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

##Additonal Dictionary

In [19]:
custom_normalization = {

    "gk": "tidak",
    "ga": "tidak",
    "nggak": "tidak",
    "tdk": "tidak",

    "bgt": "banget",
    "bgtt": "banget",

    "capek": "lelah",
    "cape": "lelah",

    "ngeri": "takut",

    "ovt": "cemas",
    "overthinking": "cemas",

    "stress": "cemas",

    "gw": "saya",
    "gue": "saya",

    "lu": "kamu",
    "loe": "kamu",

    "yg": "yang",
    "dr": "dari",

    "krn": "karena",
    "karna": "karena",

    "trs": "terus",

    "udh": "sudah",
    "udah": "sudah",

    "jg": "juga",

    "org": "orang"
}

##Typo Normalization

In [20]:
def normalize_typo(text):
    words = text.split()
    normalized_words = []

    for word in words:
        normalized_words.append(
            custom_normalization.get(
                word,
                word
            )
        )

    return " ".join(normalized_words)

##Convert Emoji

In [21]:
def convert_emoji(text):
    for emo, meaning in emoji_dict.items():
        text = text.replace(emo, meaning)
    return text

##Repeat Text

In [22]:
def normalize_repeat(text):
    text = re.sub(
        r'(.)\1{2,}',
        r'\1\1',
        text
    )

    return text

##Text Normalization

In [23]:
def normalize_text(text):
    words = text.split()
    normalized_words = []

    for word in words:
        normalized_words.append(
            combined_dict.get(
                word,
                word
            )
        )

    return " ".join(normalized_words)

##Main Pipeline

In [24]:
def preprocess_text(text):
    text = str(text)
    text = case_folding(text)
    text = normalize_typo(text)
    text = convert_emoji(text)
    text = clean_text(text)
    text = normalize_repeat(text)
    text = normalize_text(text)

    return text

In [25]:
df['clean_text'] = df['text'].swifter.apply(preprocess_text)

print(df[['text', 'clean_text']].head())

Pandas Apply: 100%|██████████| 11855/11855 [00:00<00:00, 23862.96it/s]


                                                text  \
0        lu ga lebih dri org iri yg kebetulan tolol.   
1                         benci tasik, benci jakarta   
2                         lebih takut arab di puncak   
3  buat kamu fans emyu tetap sedih dan jangan sem...   
4           hahaha tuh kan udah bangkotan bodoh pula   

                                          clean_text  
0  kamu tidak lebih dari orang iri yang kebetulan...  
1                          benci tasik benci jakarta  
2                         lebih takut arab di puncak  
3  buat kamu fan manchester united tetap sedih da...  
4        lucu apa itu kan sudah bangkotan bodoh pula  


In [26]:
df = df[df['clean_text'].str.strip() != '']

df = df.reset_index(drop=True)

#Feature Selection

In [27]:
df = df[['clean_text', 'emotion_label', 'stress_label']]

##Distribusi Label

In [28]:
df.value_counts('emotion_label')

emotion_label
anxious    2371
neutral    2371
happy      2371
sad        2371
angry      2369
Name: count, dtype: int64

# Insight

Several observations can be made from the preprocessing stage:

- The original dataset exhibited class imbalance, with the neutral class appearing more frequently than other emotional categories.
- To address this issue, synthetic data generated by Large Language Models (LLMs) was incorporated into the dataset.
- The augmentation process focused on strengthening minority classes while maintaining the original data distribution.
- After preprocessing and balancing, all emotion categories contain approximately the same number of samples, reducing potential bias toward majority classes.
- A balanced dataset is expected to support fairer model learning during the training stage.

##Convert Dataset

In [29]:
df.to_csv('Preprocessed_Dataset.csv', index=False)

# Data Dictionary

| Column | Description |
|----------|----------|
| clean_text | Cleaned text used for model training |
| emotion_label | Emotion category (angry, anxious, happy, neutral, sad) |
| stress_label | Stress score label |

# Conclusion

The preprocessing stage successfully prepared the dataset for machine learning and deep learning experiments.

Key outcomes include:

- Raw text data was cleaned and standardized through preprocessing techniques.
- Real and synthetic datasets were integrated to improve class distribution.
- Emotion categories were successfully balanced, resulting in approximately 2,370 samples per class.
- The final dataset is more suitable for training emotion classification and stress prediction models.
- The prepared dataset can now be used for tokenization, feature extraction, and model development in subsequent stages.